# ExoScout v0.1

A first machine learning baseline for distinguishing confirmed/known exoplanets from false positives using NASA TESS Objects of Interest data.

## 1. Load Data

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_data import load_data

In [2]:
df = load_data()


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/toi.csv")

## 2. Dataset Exploration

In [5]:
df.head()

,toi,tfopwg_disp,pl_orbper,pl_trandurh,pl_trandep,st_tmag,st_teff,st_logg,st_rad
0,1063.01,PC,10.066524,2.402319,839.904487,9.0689,5552.0,4.61783,0.791742
1,1064.01,CP,6.443866,1.930945,1121.951227,10.0059,4803.0,4.52079,0.737189
2,1064.02,CP,12.226560,2.190438,1206.369329,10.0059,4803.0,4.52079,0.737189
3,1065.01,KP,2.160515,1.480696,13774.879462,13.2851,5780.0,4.32928,1.200000
4,1066.01,KP,4.606184,3.121643,13793.034867,11.9707,5670.0,4.39250,1.070000


In [6]:
df.shape

(8113, 9)

## 3. Target Definition

In [7]:
filtered_df = df[df["tfopwg_disp"].isin(['CP','KP','FP'])].copy()


In [8]:
filtered_df.head()

,toi,tfopwg_disp,pl_orbper,pl_trandurh,pl_trandep,st_tmag,st_teff,st_logg,st_rad
1,1064.01,CP,6.443866,1.930945,1121.951227,10.0059,4803.0,4.52079,0.737189
2,1064.02,CP,12.226560,2.190438,1206.369329,10.0059,4803.0,4.52079,0.737189
3,1065.01,KP,2.160515,1.480696,13774.879462,13.2851,5780.0,4.32928,1.200000
4,1066.01,KP,4.606184,3.121643,13793.034867,11.9707,5670.0,4.39250,1.070000
5,1067.01,KP,3.131674,2.457920,14915.694372,13.1686,4351.0,4.43800,0.767010


In [9]:
mapping = {
    "CP": 1,
    "KP": 1,
    "FP": 0
}

filtered_df["target"] = filtered_df["tfopwg_disp"].map(mapping)

In [10]:
filtered_df.isna().sum()

toi              0
tfopwg_disp      0
pl_orbper       28
pl_trandurh      0
pl_trandep       0
st_tmag          0
st_teff         37
st_logg        217
st_rad         131
target           0
dtype: int64

In [11]:
filtered_df.dtypes

toi            float64
tfopwg_disp        str
pl_orbper      float64
pl_trandurh    float64
pl_trandep     float64
st_tmag        float64
st_teff        float64
st_logg        float64
st_rad         float64
target           int64
dtype: object

In [12]:
features = [
    "pl_orbper",
    "pl_trandurh",
    "pl_trandep",
    "st_tmag",
    "st_teff",
    "st_logg",
    "st_rad"
]

## 4. Missing Values
For v0.1, rows with missing values in the selected features were removed to keep the baseline preprocessing simple. This removed 279 rows.

In [13]:
complete_df = filtered_df.dropna(subset=features )

In [14]:
complete_df.shape


(2314, 10)

In [15]:
filtered_df.shape

(2596, 10)

## 5. Train/Test Split
The dataset was split into 80% training and 20% test data using stratification to preserve the class proportions. 

In [16]:
X = complete_df[features]
Y = complete_df["target"]

In [17]:
from sklearn.model_selection import train_test_split

In [18]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [19]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((1851, 7), (463, 7), (1851,), (463,))

In [20]:
Y_train.value_counts(normalize=True)


target
1    0.563479
0    0.436521
Name: proportion, dtype: float64

In [21]:
Y_test.value_counts(normalize=True)

target
1    0.563715
0    0.436285
Name: proportion, dtype: float64

In [22]:
from sklearn.linear_model import LogisticRegression

In [23]:
model = LogisticRegression()

## 6. Feature Scaling
The numerical features have very different scales. StandardScaler was fitted only on the training data and then applied to both training and test data to avoid data leakage.

In [24]:
from sklearn.preprocessing import StandardScaler

In [25]:
scaler= StandardScaler()

In [26]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Logistic Regression Baseline

In [27]:
model.fit(X_train_scaled, Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [28]:
Y_pred = model.predict(X_test_scaled) 

In [29]:
print(type(scaler))
print(type(X_train_scaled))
print(type(X_test_scaled))

<class 'sklearn.preprocessing._data.StandardScaler'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [30]:
Y_pred[:10]

array([0, 1, 1, 0, 1, 1, 1, 0, 1, 1])

In [31]:
Y_test.iloc[:10].to_numpy

<bound method IndexOpsMixin.to_numpy of 2231    0
1066    0
2216    1
3515    1
7076    1
1580    1
7       1
1542    1
7663    1
6435    1
Name: target, dtype: int64>

## 8. Evaluation

In [32]:
from sklearn.metrics import accuracy_score

In [33]:
accuracy = accuracy_score(Y_test, Y_pred)
print(accuracy)

0.7170626349892009


In [34]:
from sklearn.metrics import confusion_matrix

In [35]:
cm = confusion_matrix(Y_test, Y_pred)

In [36]:
cm

array([[124,  78],
       [ 53, 208]])

In [37]:
from sklearn.metrics import classification_report

In [38]:
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.70      0.61      0.65       202
           1       0.73      0.80      0.76       261

    accuracy                           0.72       463
   macro avg       0.71      0.71      0.71       463
weighted avg       0.72      0.72      0.71       463



## 9. v0.1 Results

The Logistic Regression baseline achieved approximately 71.7% accuracy on the held-out test set.

For the planet class:
- Precision: ~73%
- Recall: ~80%
- F1-score: ~76%

The model correctly identified 208 planets and missed 53. It also classified 78 false positives as planets.

The main weakness of this baseline is therefore its ability to reject false positives: recall for the false-positive class was only about 61%.

## 10. Limitations and Next Steps

Current limitations:
- 279 rows with missing values were discarded.
- Only one model has been tested.
- Evaluation is based on a single train/test split.
- Only tabular TOI features are currently used.
- No feature engineering or hyperparameter tuning has been performed.

Possible improvements for future versions:
- Replace row deletion with missing-value imputation.
- Compare Logistic Regression with other models.
- Use cross-validation for more robust evaluation.
- Investigate prediction errors.
- Explore feature engineering.
- Eventually incorporate TESS light-curve data.

# ExoScout v0.2

## 1. Missingness Analysis by Class

In [39]:
missing_by_class = (
    filtered_df
    .groupby("target")[features]
    .apply(lambda group: group.isna().mean())
)

missing_by_class

,pl_orbper,pl_trandurh,pl_trandep,st_tmag,st_teff,st_logg,st_rad
target,,,,,,,
0,0.008765,0.0,0.0,0.0,0.029482,0.162550,0.096414
1,0.012677,0.0,0.0,0.0,0.000000,0.009694,0.007457


### Finding

Missing values are not evenly distributed across the two classes.

In particular, `st_logg` is missing in about 16.3% of false positives but only about 1.0% of planets, while `st_rad` is missing in about 9.6% of false positives and 0.7% of planets.

This suggests that dropping every row containing missing values may disproportionately remove false positives and potentially bias the training dataset.

In [40]:
complete_by_class = (
    filtered_df
    .assign(is_complete=filtered_df[features].notna().all(axis=1))
    .groupby("target")["is_complete"]
    .mean()
)

complete_by_class

target
0    0.804781
1    0.972409
Name: is_complete, dtype: float64

### Complete-case analysis

Using complete cases only retains about 80.5% of false positives but 97.2% of planets.

Therefore, the v0.1 `dropna()` strategy disproportionately removes false positives. This can alter the class distribution and potentially bias both training and evaluation.

For v0.2, missing values will therefore be handled through imputation rather than row deletion.

In [41]:
X = filtered_df[features]
Y = filtered_df["target"]

For v0.2, the model uses the full filtered dataset, including rows with missing feature values. Missing values will be handled with imputation instead of dropping rows.

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [43]:
X_train.shape, X_test.shape

((2076, 7), (520, 7))

In [44]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

In [45]:
import numpy as np

np.isnan(X_train_imputed).sum(), np.isnan(X_test_imputed).sum()

(np.int64(0), np.int64(0))

In [46]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [47]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [48]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion matrix:")
print(cm)
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.676923076923077
Confusion matrix:
[[160  91]
 [ 77 192]]

              precision    recall  f1-score   support

           0       0.68      0.64      0.66       251
           1       0.68      0.71      0.70       269

    accuracy                           0.68       520
   macro avg       0.68      0.68      0.68       520
weighted avg       0.68      0.68      0.68       520



In [49]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])


In [50]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [51]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [52]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [54]:
results = cross_validate(
    pipeline,
    X,
    Y,
    cv=cv,
    scoring=scoring
)

In [55]:
results

{'fit_time': array([0.04074502, 0.01486325, 0.01792884, 0.0252552 , 0.01755691]),
 'score_time': array([0.04229689, 0.01858878, 0.01748133, 0.01779675, 0.02153492]),
 'test_accuracy': array([0.68076923, 0.68208092, 0.6743738 , 0.67630058, 0.71098266]),
 'test_precision': array([0.66885246, 0.67697595, 0.68679245, 0.68656716, 0.70774648]),
 'test_recall': array([0.75836431, 0.73507463, 0.67910448, 0.68656716, 0.75      ]),
 'test_f1': array([0.71080139, 0.70483005, 0.68292683, 0.68656716, 0.72826087])}

In [56]:
accuracy_mean = results["test_accuracy"].mean()
accuracy_std = results["test_accuracy"].std()

accuracy_mean, accuracy_std

(np.float64(0.6849014376760042), np.float64(0.01334216295051377))

In [58]:
precision_mean = results["test_precision"].mean()
precision_std = results["test_precision"].std()

recall_mean = results["test_recall"].mean()
recall_std = results["test_recall"].std()

f1_mean = results["test_f1"].mean()
f1_std = results["test_f1"].std()

precision_mean, precision_std, recall_mean, recall_std, f1_mean, f1_std

(np.float64(0.6853868999832217),
 np.float64(0.013018842283727822),
 np.float64(0.7218221161848749),
 np.float64(0.032780062710046604),
 np.float64(0.7026772620816201),
 np.float64(0.016581748093851403))

### Logistic Regression — 5-Fold Cross-Validation

Using median imputation, standard scaling and Logistic Regression:

- Accuracy: 68.5% ± 1.3%
- Precision (planet): 68.5% ± 1.3%
- Recall (planet): 72.2% ± 3.3%
- F1-score (planet): 70.3% ± 1.7%

These results suggest that the single-split v0.1 accuracy of ~71.7% was likely somewhat optimistic. Cross-validation provides a more stable estimate of baseline performance.

### Experiment: Missingness Indicators

The previous analysis showed that missing values are not evenly distributed between the two classes.

For example, `st_logg` and `st_rad` are missing much more frequently among false positives than among confirmed/known planets.

In the previous pipeline, missing values were replaced with the median. However, after imputation, the model could no longer distinguish between an original value and a value that had been inserted by the imputer.

To preserve this information, this experiment adds a binary missingness indicator for features containing missing values.

For each affected feature, the indicator records whether the original value was missing:

- `0` → the original value was available
- `1` → the original value was missing and was imputed

The rest of the pipeline remains unchanged:

1. Median imputation
2. Missingness indicators
3. Standard scaling
4. Logistic Regression
5. 5-fold stratified cross-validation

By keeping the model and cross-validation setup unchanged, we can isolate the effect of adding missingness information.

In [59]:
pipeline_with_indicators = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

In [61]:
results_with_indicators = cross_validate(
    pipeline_with_indicators,
    X,
    Y,
    cv=cv,
    scoring=scoring
)

In [62]:
results_with_indicators["test_accuracy"]

array([0.73076923, 0.71868979, 0.73603083, 0.72447013, 0.76685934])

In [63]:
indicator_accuracy_mean = results_with_indicators["test_accuracy"].mean()
indicator_accuracy_std = results_with_indicators["test_accuracy"].std()

indicator_precision_mean = results_with_indicators["test_precision"].mean()
indicator_precision_std = results_with_indicators["test_precision"].std()

indicator_recall_mean = results_with_indicators["test_recall"].mean()
indicator_recall_std = results_with_indicators["test_recall"].std()

indicator_f1_mean = results_with_indicators["test_f1"].mean()
indicator_f1_std = results_with_indicators["test_f1"].std()

(
    indicator_accuracy_mean,
    indicator_accuracy_std,
    indicator_precision_mean,
    indicator_precision_std,
    indicator_recall_mean,
    indicator_recall_std,
    indicator_f1_mean,
    indicator_f1_std
)

(np.float64(0.7353638654216688),
 np.float64(0.01679415943622169),
 np.float64(0.7243144020696367),
 np.float64(0.0184633268775179),
 np.float64(0.7889502302613327),
 np.float64(0.03399449644640934),
 np.float64(0.754692573085974),
 np.float64(0.01720650319950119))

### Result

Adding missingness indicators produced a consistent improvement across all 5 folds.

Compared with median imputation alone:

- Accuracy increased from 68.5% ± 1.3% to 73.5% ± 1.7%
- Precision increased from 68.5% ± 1.3% to 72.4% ± 1.8%
- Recall increased from 72.2% ± 3.3% to 78.9% ± 3.4%
- F1-score increased from 70.3% ± 1.7% to 75.5% ± 1.7%

This suggests that the pattern of missing values contains useful predictive information and should be preserved rather than discarded.

### Experiment Log

The goal of this table is to compare preprocessing choices under the same 5-fold stratified cross-validation setup.

In [64]:
import pandas as pd

experiment_log = pd.DataFrame({
    "Experiment": [
        "Median imputation",
        "Median imputation + missing indicators"
    ],
    "Accuracy": [
        accuracy_mean,
        indicator_accuracy_mean
    ],
    "Precision": [
        precision_mean,
        indicator_precision_mean
    ],
    "Recall": [
        recall_mean,
        indicator_recall_mean
    ],
    "F1": [
        f1_mean,
        indicator_f1_mean
    ]
})

experiment_log

,Experiment,Accuracy,Precision,Recall,F1
0,Median imputation,0.684901,0.685387,0.721822,0.702677
1,Median imputation + missing indicators,0.735364,0.724314,0.788950,0.754693


In [65]:
experiment_log.round(3)

,Experiment,Accuracy,Precision,Recall,F1
0,Median imputation,0.685,0.685,0.722,0.703
1,Median imputation + missing indicators,0.735,0.724,0.789,0.755


## v0.2 Conclusion

The v0.2 experiments showed that missingness is strongly associated with the target class.

Replacing missing values with median imputation alone produced an average accuracy of 68.5% under 5-fold cross-validation.

Preserving missingness information through binary indicators improved average accuracy to 73.5%, with improvements across all evaluated metrics and all five folds.

This suggests that missing-data patterns contain useful predictive information in the TOI dataset and should be explicitly preserved in future models.